In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (CLOX_v3)
#   Read EXTR isobar ZM (200-yr clim) and Hindcast h3 (hybrid),
#   interpolate hindcast CLOX to EXTR isobar lev grid,
#   then cache TWO products for later blocks:
#
#   (1) 60–90°N polar-cap mean (zonal mean already)
#       - EXTR clim series (time, lev)
#       - HIND series for needed years (time, lev)
#
#   (2) 82°N zonal mean
#       - EXTR clim series (time, lev)  [from EXTR ZM]
#       - HIND series for needed years (time, lev)
#
# OUTPUT:
#   CLOX_EXTR_pcap_60_90N.nc      (time, lev)
#   CLOX_HIND_pcap_60_90N.nc      (time, lev)
#   CLOX_EXTR_82N_zm.nc           (time, lev)
#   CLOX_HIND_82N_zm.nc           (time, lev)
#
# NOTES:
#   * lev in EXTR is treated as pressure in hPa.
#   * Hindcast lev is hybrid-sigma; p_mid = hyam*P0 + hybm*PS.
#   * log-p interpolation to EXTR plevs.
# ============================================================

import os, glob
import numpy as np
import xarray as xr

ROOT_OUT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_CLOX"
os.makedirs(ROOT_OUT, exist_ok=True)

EXTR_ZM_PATH = (
    "/mnt/backup_ETH/extr_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300.CLO.isobar.zm.nc"
)

H3_DIR = (
    "/mnt/backup_ETH/EXTR_2000/EXTR_2000/"
    "BWCN.e122.f19_g16.002/atm/hist"
)

OUT_EXTR_PCAP = os.path.join(ROOT_OUT, "CLOX_EXTR_pcap_60_90N.nc")
OUT_HIND_PCAP = os.path.join(ROOT_OUT, "CLOX_HIND_pcap_60_90N.nc")
OUT_EXTR_82N  = os.path.join(ROOT_OUT, "CLOX_EXTR_82N_zm.nc")
OUT_HIND_82N  = os.path.join(ROOT_OUT, "CLOX_HIND_82N_zm.nc")

YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)
YEARS_NEEDED  = np.unique(np.concatenate([YEARS_SPECIAL, YEARS_SPECIAL-1]))
LAT_TGT = 82.0


def _lev_hpa_to_pa(lev_vals):
    return np.asarray(lev_vals, dtype=float) * 100.0

def compute_pressure_mid(ds):
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]
    P0   = ds["P0"]    # scalar Pa
    PS   = ds["PS"]    # (time, lat, lon) Pa
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_to_plev_logp(var_hyb, p_hyb, plev_target_pa):
    p_tgt = np.asarray(plev_target_pa, dtype=float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt.shape, np.nan, dtype=float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending p
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use,
                         left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        var_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def open_h3_for_years(years):
    fps = []
    for y in years:
        yy = f"{y:04d}"
        fps.extend(sorted(glob.glob(os.path.join(H3_DIR, f"*.cam.h3.{yy}-*.nc"))))
    return sorted(list(dict.fromkeys(fps)))

def main_blockA_CLOX_v3():

    # ===== 1) EXTR climatology (isobar ZM) =====
    print("[BlockA_CLOX_v3] Reading EXTR ZM:", EXTR_ZM_PATH)
    ds_extr = xr.open_dataset(EXTR_ZM_PATH)
    varname_extr = "CLOX" if "CLOX" in ds_extr else "CLO"
    da_extr = ds_extr[varname_extr]          # (time, lev, lat)
    lev_extr_hpa = da_extr["lev"].values
    plev_target_pa = _lev_hpa_to_pa(lev_extr_hpa)

    # EXTR polar cap 60-90N
    clox_extr_pcap = da_extr.sel(lat=slice(60,90)).mean("lat")  # (time, lev)
    clox_extr_pcap = clox_extr_pcap.transpose("time","lev")
    clox_extr_pcap.name="CLOX_EXTR_pcap_60_90N"
    clox_extr_pcap.attrs.update(units="mol/mol",
                                long_name="EXTR climatology CLOX polar-cap mean (60–90N)")
    clox_extr_pcap.to_netcdf(OUT_EXTR_PCAP)
    print("[BlockA_CLOX_v3] Saved:", OUT_EXTR_PCAP)

    # EXTR 82N zonal mean (EXTR already ZM)
    clox_extr_82 = da_extr.sel(lat=LAT_TGT, method="nearest")
    clox_extr_82 = clox_extr_82.transpose("time","lev")
    clox_extr_82.name="CLOX_EXTR_82N_zm"
    clox_extr_82.attrs.update(units="mol/mol",
                              long_name="EXTR climatology CLOX zonal-mean at 82N")
    clox_extr_82.to_netcdf(OUT_EXTR_82N)
    print("[BlockA_CLOX_v3] Saved:", OUT_EXTR_82N)

    ds_extr.close()

    # ===== 2) Hindcast h3 (hybrid -> EXTR isobar grid) =====
    h3_files = open_h3_for_years(YEARS_NEEDED)
    print(f"[BlockA_CLOX_v3] Opening needed h3 years: {YEARS_NEEDED.tolist()}")
    print(f"[BlockA_CLOX_v3] Found {len(h3_files)} h3 files.")

    pieces_pcap=[]
    pieces_82=[]

    for fp in h3_files:
        print("  -", os.path.basename(fp))
        ds = xr.open_dataset(fp)
        if "CLOX" not in ds:
            ds.close()
            continue

        clox_hyb = ds["CLOX"]               # (time, lev, lat, lon)
        p_mid = compute_pressure_mid(ds)    # (time, lev, lat, lon)

        # interpolate to EXTR plevs
        clox_iso = interp_to_plev_logp(clox_hyb, p_mid, plev_target_pa)
        clox_iso = clox_iso.rename({"plev":"lev"})
        clox_iso = clox_iso.assign_coords(lev=("lev", lev_extr_hpa))
        clox_iso["lev"].attrs["units"]="hPa"

        # 60–90N polar cap mean (zonal mean first)
        clox_zm = clox_iso.mean("lon")  # (time, lev, lat)
        clox_pcap = clox_zm.sel(lat=slice(60,90)).mean("lat")  # (time, lev)

        # 82N zonal mean
        clox_82 = clox_zm.sel(lat=LAT_TGT, method="nearest")  # (time, lev)

        yrs = clox_pcap.time.dt.year.values.astype(int)
        mask = np.isin(yrs, YEARS_NEEDED)
        if np.any(mask):
            pieces_pcap.append(clox_pcap.sel(time=mask))
            pieces_82.append(clox_82.sel(time=mask))

        ds.close()

    hind_pcap = xr.concat(pieces_pcap, dim="time").sortby("time").transpose("time","lev")
    hind_82   = xr.concat(pieces_82,   dim="time").sortby("time").transpose("time","lev")

    hind_pcap.name="CLOX_HIND_pcap_60_90N"
    hind_pcap.attrs.update(units="mol/mol",
                           long_name="Hindcast CLOX polar-cap mean (60–90N), on EXTR isobar levs")
    hind_pcap.to_netcdf(OUT_HIND_PCAP)
    print("[BlockA_CLOX_v3] Saved:", OUT_HIND_PCAP)

    hind_82.name="CLOX_HIND_82N_zm"
    hind_82.attrs.update(units="mol/mol",
                         long_name="Hindcast CLOX zonal-mean at 82N, on EXTR isobar levs")
    hind_82.to_netcdf(OUT_HIND_82N)
    print("[BlockA_CLOX_v3] Saved:", OUT_HIND_82N)

    print("[BlockA_CLOX_v3] Done.")

if __name__ == "__main__":
    main_blockA_CLOX_v3()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (CLOX_v5)
#
# For BOTH regions:
#   R1) 82°N zonal-mean CLOX
#   R2) 60–90°N polar-cap weighted-mean CLOX
#
# Each pressure level has its own figure:
#   - special years (0008/0013/0014/0019), stitched Oct–Sep
#   - EXTR all-years climatology (stitched Oct–Sep) + ±1σ envelope
#   - EXTR low25% climatology (Y-1 Oct–Dec + Y Jan–Sep) + ±1σ envelope
#
# X-axis: day index (0..364), month ticks Oct–Sep (avoid cftime plotting)
#
# INPUT (from BlockA):
#   1) 82N:
#       CLOX_EXTR_82N_zm.nc   (time, lev[hPa])
#       CLOX_HIND_82N_zm.nc   (time, lev[hPa])
#   2) pcap 60-90N:
#       CLOX_pcap_EXTR_60_90N_lev_time.nc (time, lev[hPa])
#       CLOX_pcap_HIND_60_90N_lev_time.nc (time, lev[hPa])  # assumed
#   3) O3_lowest25pct_years.txt
#
# OUTPUT:
#   CLOX_<region>_<lev>hPa_special_vs_extr_climatology.png/pdf
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

ROOT_O3   = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
ROOT_CLOX = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_CLOX"
os.makedirs(ROOT_CLOX, exist_ok=True)

# ---- 82°N cached data (from BlockA) ----
CLOX_EXTR_82N_NC = os.path.join(ROOT_CLOX, "CLOX_EXTR_82N_zm.nc")
CLOX_HIND_82N_NC = os.path.join(ROOT_CLOX, "CLOX_HIND_82N_zm.nc")

# ---- 60–90°N polar-cap cached data (from BlockA log) ----
CLOX_EXTR_PCAP_NC = os.path.join(ROOT_CLOX, "CLOX_EXTR_pcap_60_90N.nc")
# hind pcap filename assumed; adjust if your BlockA used a different name
CLOX_HIND_PCAP_NC = os.path.join(ROOT_CLOX, "CLOX_HIND_pcap_60_90N.nc")

LOW25_TXT = os.path.join(ROOT_O3, "O3_lowest25pct_years.txt")

# previous-year stitched length for Oct–Dec
N_PREV = 91

# special hindcast years
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

# target pressure levels (hPa)
PLEV_SEL = [10, 50, 100]

# month ticks for Oct–Sep on day-index axis
XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"]

def get_level_da(da_lev_time, target_hpa):
    """Select nearest pressure level (lev in hPa)."""
    da_p = da_lev_time.sel(lev=target_hpa, method="nearest")
    lev_actual = float(da_p.lev.values)
    return da_p, lev_actual

def _select_year(da, year):
    return da.sel(time=da.time.dt.year == year)

def stitch_prev_oct_to_curr_sep(da_1lev, year):
    """Oct–Dec of (year-1) + Jan–Sep of (year)."""
    prev = _select_year(da_1lev, year-1)
    curr = _select_year(da_1lev, year)

    if prev.sizes["time"] == 0 or curr.sizes["time"] == 0:
        return xr.zeros_like(da_1lev.isel(time=slice(0, 0)))

    prev_oct_dec = prev.sel(time=prev.time.dt.month.isin([10,11,12]))
    curr_jan_sep = curr.sel(time=curr.time.dt.month.isin([1,2,3,4,5,6,7,8,9]))
    return xr.concat([prev_oct_dec, curr_jan_sep], dim="time")

def build_extr_climatologies(var_extr_1lev, lowest25_years):
    """
    Same logic as your U BlockB:
      - all-years daily climatology
      - low25% daily climatology using Y-1 Oct–Dec + Y Jan–Sep fix
    Return stitched Oct–Sep mean/std arrays of length 365.
    """
    n_days = int(var_extr_1lev.time.dt.dayofyear.max())

    # all-years daily climatology
    clim_all_mean = var_extr_1lev.groupby("time.dayofyear").mean("time")
    clim_all_std  = var_extr_1lev.groupby("time.dayofyear").std("time")

    # low25% daily climatology (FIX)
    base_low25_cur  = var_extr_1lev.sel(time=var_extr_1lev.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr_1lev.sel(time=var_extr_1lev.time.dt.year.isin(lowest25_years - 1))

    clim_low25_cur_mean  = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_cur_std   = base_low25_cur.groupby("time.dayofyear").std("time")
    clim_low25_prev_mean = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_prev_std  = base_low25_prev.groupby("time.dayofyear").std("time")

    clim_low25_full_mean = clim_low25_cur_mean.copy()
    clim_low25_full_std  = clim_low25_cur_std.copy()

    oct_dec_doys = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)
    clim_low25_full_mean.loc[dict(dayofyear=oct_dec_doys)] = (
        clim_low25_prev_mean.sel(dayofyear=oct_dec_doys).values
    )
    clim_low25_full_std.loc[dict(dayofyear=oct_dec_doys)] = (
        clim_low25_prev_std.sel(dayofyear=oct_dec_doys).values
    )

    clim_all_mean = np.array(clim_all_mean);  clim_all_std  = np.array(clim_all_std)
    clim_low_mean = np.array(clim_low25_full_mean); clim_low_std = np.array(clim_low25_full_std)

    # stitched Oct–Sep
    all_prev_mean = clim_all_mean[n_days-N_PREV:n_days]
    all_prev_std  = clim_all_std[n_days-N_PREV:n_days]
    all_cur_mean  = clim_all_mean[0:n_days-N_PREV]
    all_cur_std   = clim_all_std[0:n_days-N_PREV]
    all_comp_mean = np.concatenate([all_prev_mean, all_cur_mean])
    all_comp_std  = np.concatenate([all_prev_std,  all_cur_std])

    low_prev_mean = clim_low_mean[n_days-N_PREV:n_days]
    low_prev_std  = clim_low_std[n_days-N_PREV:n_days]
    low_cur_mean  = clim_low_mean[0:n_days-N_PREV]
    low_cur_std   = clim_low_std[0:n_days-N_PREV]
    low_comp_mean = np.concatenate([low_prev_mean, low_cur_mean])
    low_comp_std  = np.concatenate([low_prev_std,  low_cur_std])

    return all_comp_mean, all_comp_std, low_comp_mean, low_comp_std, n_days

def plot_one_level_special_vs_clim(var_extr_1lev, var_hind_1lev, lowest25_years,
                                   lev_actual_hpa, region_tag, region_title,
                                   out_png, out_pdf):
    """One figure for one region and one level."""
    all_mean, all_std, low_mean, low_std, n_days = \
        build_extr_climatologies(var_extr_1lev, lowest25_years)

    years_hind = np.unique(var_hind_1lev.time.dt.year.values.astype(int))
    missing_special = YEARS_SPECIAL[~np.isin(YEARS_SPECIAL, years_hind)]
    if len(missing_special) > 0:
        print(f"⚠️ Missing special years in hind ({region_tag}, {lev_actual_hpa:.1f} hPa):",
              missing_special)

    # plotting style
    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)
    x_full = np.arange(n_days)

    # special years stitched
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    handles_years = []
    for i, y in enumerate(YEARS_SPECIAL):
        if y not in years_hind:
            continue
        stitched = stitch_prev_oct_to_curr_sep(var_hind_1lev, y)
        if stitched.sizes["time"] != n_days:
            print(f"⚠️ stitched length mismatch for {y:04d} ({region_tag}, {lev_actual_hpa:.1f} hPa)")
            continue

        ax.plot(x_full, stitched.values,
                color=colors_spec[i], lw=1.5, ls="-",
                label=f"Year {y:04d}")
        handles_years.append(Line2D([0],[0], color=colors_spec[i], lw=1.5,
                                    label=f"Year {y:04d}"))

    # EXTR all climatology ±1σ
    ax.fill_between(x_full, all_mean-all_std, all_mean+all_std,
                    color="0.85", alpha=0.9, linewidth=0, zorder=0)
    ax.plot(x_full, all_mean, color="black", lw=1.8, ls="--",
            label="EXTR climatology (all years)")

    # EXTR low25 climatology ±1σ
    ax.fill_between(x_full, low_mean-low_std, low_mean+low_std,
                    facecolor="none", edgecolor="0.5", hatch="///",
                    linewidth=0.0, zorder=1)
    ax.plot(x_full, low_mean, color="black", lw=2.0, ls="-",
            label="EXTR low-ozone composite")

    # axes
    ax.set_xlim(0, n_days)
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)
    ax.set_xlabel("Seasonal months (Oct–Sep)")
    ax.set_ylabel(f"CLOX ({region_title}), {lev_actual_hpa:.1f} hPa (ppb)")
    ax.set_title(f"CLOX {region_title} — {lev_actual_hpa:.1f} hPa")

    # legend
    patch_all = Patch(facecolor="0.85", edgecolor="none", label="EXTR all-years ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///",
                      label="EXTR low25% ±1σ")
    line_all = Line2D([0],[0], color="black", lw=1.8, ls="--",
                      label="EXTR climatology (all years)")
    line_low = Line2D([0],[0], color="black", lw=2.0, ls="-",
                      label="EXTR low-ozone composite")

    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)
    print("[BlockB_CLOX_v5] Saved:", out_png)
    print("                    ", out_pdf)

def main_blockB_CLOX_v5():

    # ---- load low25 O3 years ----
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockB_CLOX_v5] Lowest 25% O3 years:", lowest25_years)

    # ---- define two regions to process ----
    regions = [
        dict(
            tag="82N",
            title="at 82°N (zonal mean)",
            extr_nc=CLOX_EXTR_82N_NC,
            hind_nc=CLOX_HIND_82N_NC
        ),
        dict(
            tag="pcap60_90N",
            title="polar cap 60–90°N (weighted mean)",
            extr_nc=CLOX_EXTR_PCAP_NC,
            hind_nc=CLOX_HIND_PCAP_NC
        ),
    ]

    for reg in regions:
        print(f"\n[BlockB_CLOX_v5] ===== Region: {reg['tag']} =====")
        print("  EXTR:", reg["extr_nc"])
        print("  HIND:", reg["hind_nc"])

        da_extr = xr.load_dataarray(reg["extr_nc"])  # (time, lev[hPa])
        da_hind = xr.load_dataarray(reg["hind_nc"])

        # to ppb
        da_extr_ppb = da_extr * 1e9
        da_hind_ppb = da_hind * 1e9

        for plev in PLEV_SEL:
            var_extr_1lev, lev_actual = get_level_da(da_extr_ppb, plev)
            var_hind_1lev, _          = get_level_da(da_hind_ppb, plev)

            out_png = os.path.join(
                ROOT_CLOX,
                f"CLOX_{reg['tag']}_{int(round(lev_actual))}hPa_special_vs_extr_climatology.png"
            )
            out_pdf = os.path.join(
                ROOT_CLOX,
                f"CLOX_{reg['tag']}_{int(round(lev_actual))}hPa_special_vs_extr_climatology.pdf"
            )

            print(f"[BlockB_CLOX_v5] Plotting ~{plev} hPa (actual {lev_actual:.2f} hPa)")
            plot_one_level_special_vs_clim(
                var_extr_1lev=var_extr_1lev,
                var_hind_1lev=var_hind_1lev,
                lowest25_years=lowest25_years,
                lev_actual_hpa=lev_actual,
                region_tag=reg["tag"],
                region_title=reg["title"],
                out_png=out_png,
                out_pdf=out_pdf
            )

    print("\n[BlockB_CLOX_v5] Done.")

if __name__ == "__main__":
    main_blockB_CLOX_v5()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (CLOX_v7_special)
#
# Compute time–height anomalies for SPECIAL hindcast years:
#   - raw anomaly (ppb) vs baseline ALL climatology
#   - raw anomaly (ppb) vs baseline LOW25 climatology
#   - standardized anomaly (unitless) vs BOTH baselines
#
# For BOTH regions:
#   R1) 82°N zonal mean
#   R2) polar cap 60–90°N weighted mean
#
# Baselines built from EXTR (200 yrs):
#   - ALL-year daily climatology (15-day running-mean seasonal cycle)
#   - LOW25 daily climatology using Y-1 Oct–Dec + Y Jan–Sep fix
#
# Significance masks for EACH baseline and EACH anomaly type:
#   - t-test
#   - bootstrap CI
#
# INPUT (from BlockA):
#   CLOX_EXTR_pcap_60_90N.nc      (time, lev[hPa])
#   CLOX_HIND_pcap_60_90N.nc      (time, lev[hPa])
#   CLOX_EXTR_82N_zm.nc           (time, lev[hPa])
#   CLOX_HIND_82N_zm.nc           (time, lev[hPa])
#   O3_lowest25pct_years.txt
#
# OUTPUT:
#   CLOX_vert_anom_special.nc
#     coords:
#       ref_year, lev, time_index(0..364), dayofyear(for Oct–Sep stitch)
#     vars (all ppb unless noted):
#       anom_all_<region>, anom_low25_<region>
#       std_anom_all_<region>, std_anom_low25_<region>   (unitless)
#       clim_all_comp_<region>, clim_low25_comp_<region>
#       t_sig_all_<region>, b_sig_all_<region>
#       t_sig_low25_<region>, b_sig_low25_<region>
#       t_sig_std_all_<region>, b_sig_std_all_<region>
#       t_sig_std_low25_<region>, b_sig_std_low25_<region>
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

ROOT_O3   = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
ROOT_CLOX = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_CLOX"
os.makedirs(ROOT_CLOX, exist_ok=True)

IN_EXTR_PCAP = os.path.join(ROOT_CLOX, "CLOX_EXTR_pcap_60_90N.nc")
IN_HIND_PCAP = os.path.join(ROOT_CLOX, "CLOX_HIND_pcap_60_90N.nc")
IN_EXTR_82N  = os.path.join(ROOT_CLOX, "CLOX_EXTR_82N_zm.nc")
IN_HIND_82N  = os.path.join(ROOT_CLOX, "CLOX_HIND_82N_zm.nc")

LOW25_TXT = os.path.join(ROOT_O3, "O3_lowest25pct_years.txt")
OUT_NC    = os.path.join(ROOT_CLOX, "CLOX_vert_anom_special.nc")

# stitch setting
N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

RUNMEAN_DAYS = 15
ALPHA = 0.05
NBOOT = 5000

# ---------------- debug helpers ----------------
def _debug_da(name, da):
    print(f"\n[DEBUG] {name}")
    print("  dims :", da.dims)
    print("  shape:", da.shape)
    print("  coords:", list(da.coords))
    if "lat" in da.coords:
        print("  lat coord:", da.lat.values)
    print("  lev range:", float(da.lev.min()), "→", float(da.lev.max()))
    print("  time range:", da.time.values[0], "→", da.time.values[-1])

def strip_nondim_coords(da):
    drop = [c for c in da.coords if c not in da.dims]
    if drop:
        print("[strip] dropping non-dim coords:", drop)
        da = da.drop_vars(drop)
    return da

# ---------------- seasonal cycle ----------------
def daily_seasonal_cycle_rm(da):
    """
    da: (time, lev)
    return clim_rm: (doy, lev) 15-day running mean seasonal cycle
    """
    doy = da.time.dt.dayofyear
    da2 = da.assign_coords(doy=("time", doy.values))
    clim = da2.groupby("doy").mean("time")
    clim_rm = clim.rolling(doy=RUNMEAN_DAYS, center=True, min_periods=1).mean()
    return clim_rm

def daily_std(da):
    doy = da.time.dt.dayofyear
    da2 = da.assign_coords(doy=("time", doy.values))
    return da2.groupby("doy").std("time")

# ---------------- baseline builder ----------------
def build_low25_daily_mean_std(extr, lowest25_years):
    """
    extr: (time, lev)
    Build LOW25 daily climatology WITH prev-year fix:
      - mean_low25_full(doy, lev)
      - std_low25_full(doy, lev)
    """
    n_days = int(extr.time.dt.dayofyear.max())

    base_cur  = extr.sel(time=extr.time.dt.year.isin(lowest25_years))
    base_prev = extr.sel(time=extr.time.dt.year.isin(lowest25_years - 1))

    mean_cur  = base_cur.groupby("time.dayofyear").mean("time")
    std_cur   = base_cur.groupby("time.dayofyear").std("time")

    mean_prev = base_prev.groupby("time.dayofyear").mean("time")
    std_prev  = base_prev.groupby("time.dayofyear").std("time")

    # copy current, replace Oct–Dec with previous-year stats
    mean_full = mean_cur.copy()
    std_full  = std_cur.copy()

    oct_dec_doys = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)
    mean_full.loc[dict(dayofyear=oct_dec_doys)] = mean_prev.sel(dayofyear=oct_dec_doys).values
    std_full.loc[dict(dayofyear=oct_dec_doys)]  = std_prev.sel(dayofyear=oct_dec_doys).values

    return mean_full, std_full

# ---------------- stitch helpers ----------------
def _select_year(da, year):
    return da.sel(time=da.time.dt.year == year)

def stitch_prev_oct_to_curr_sep_2d(da_time_lev, year):
    """
    da_time_lev: (time, lev)
    return stitched (time=365, lev) for seasonal year Oct–Sep
    """
    prev = _select_year(da_time_lev, year-1)
    curr = _select_year(da_time_lev, year)

    if prev.sizes["time"] == 0 or curr.sizes["time"] == 0:
        print(f"[stitch] year={year:04d} missing prev/curr -> None")
        return None

    prev_oct_dec = prev.sel(time=prev.time.dt.month.isin([10,11,12]))
    curr_jan_sep = curr.sel(time=curr.time.dt.month.isin([1,2,3,4,5,6,7,8,9]))
    stitched = xr.concat([prev_oct_dec, curr_jan_sep], dim="time")

    if stitched.sizes["time"] != 365:
        print(f"[stitch] year={year:04d} stitched length={stitched.sizes['time']} (expect 365)")
    return stitched

# ---------------- significance ----------------
def bootstrap_ci(data, nboot=NBOOT, alpha=ALPHA, rng=None):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan
    if rng is None:
        rng = np.random.default_rng()
    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)
    lo = np.percentile(means, 100*alpha/2.0)
    hi = np.percentile(means, 100*(1-alpha/2.0))
    return lo, hi

def compute_significance_for_baseline(
    base_anom,   # (time, lev) DataArray
    anom_ref,    # (lev, time_index) np.array
    doy_base,    # (time,) int
    doy_comp,    # (time_index,) int
    alpha=ALPHA,
    nboot=NBOOT,
):
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue
        day_samples = base_vals[mask_day, :]  # (n_samp, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]

            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig

# ---------------- main ----------------
def process_region(extr, hind, lowest25_years, region_tag):
    """
    extr/hind: (time, lev) mol/mol
    return dict of arrays for this region.
    """
    print(f"\n[BlockC_CLOX_v7] ==== Region: {region_tag} ====")
    _debug_da(f"extr_{region_tag}(raw)", extr)
    _debug_da(f"hind_{region_tag}(raw)", hind)

    extr = strip_nondim_coords(extr)
    hind = strip_nondim_coords(hind)

    # to ppb for storage (but keep mol/mol for sig calc)
    extr_ppb = extr * 1e9
    hind_ppb = hind * 1e9

    n_days = int(extr.time.dt.dayofyear.max())
    lev = extr.lev.values
    lev_n = lev.size

    doy_extr = extr.time.dt.dayofyear.values.astype(int)

    # ---- ALL baseline (daily mean + std) ----
    clim_all_daily_mean = extr.groupby("time.dayofyear").mean("time")   # (doy, lev)
    clim_all_daily_std  = extr.groupby("time.dayofyear").std("time")    # (doy, lev)

    # 15-day RM seasonal cycle for baseline mean
    clim_all_cycle_rm = clim_all_daily_mean.rolling(
        dayofyear=RUNMEAN_DAYS, center=True, min_periods=1
    ).mean()

    # ---- LOW25 baseline (daily mean + std with prev-year fix) ----
    clim_low25_daily_mean, clim_low25_daily_std = \
        build_low25_daily_mean_std(extr, lowest25_years)

    clim_low25_cycle_rm = clim_low25_daily_mean.rolling(
        dayofyear=RUNMEAN_DAYS, center=True, min_periods=1
    ).mean()

    # ---- composite DOY sequence (Oct–Sep stitch) ----
    doy_comp = np.concatenate(
        [
            np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
            np.arange(1, n_days - N_PREV + 1, dtype=int),
        ]
    )
    t_len = doy_comp.size
    print("[DEBUG] doy_comp head/tail:", doy_comp[:5], "...", doy_comp[-5:])

    # ---- stitched baseline mean/std (mol/mol) ----
    clim_all_comp_mean = clim_all_cycle_rm.sel(dayofyear=doy_comp)         # (time_index, lev)
    clim_all_comp_std  = clim_all_daily_std.sel(dayofyear=doy_comp)

    clim_low_comp_mean = clim_low25_cycle_rm.sel(dayofyear=doy_comp)
    clim_low_comp_std  = clim_low25_daily_std.sel(dayofyear=doy_comp)

    clim_all_comp_vals = clim_all_comp_mean.transpose("lev","dayofyear").values
    clim_low_comp_vals = clim_low_comp_mean.transpose("lev","dayofyear").values
    std_all_comp_vals  = clim_all_comp_std.transpose("lev","dayofyear").values
    std_low_comp_vals  = clim_low_comp_std.transpose("lev","dayofyear").values

    # ---- base anomalies for significance (mol/mol) ----
    # ALL baseline
    clim_all_on_extr = clim_all_cycle_rm.sel(dayofyear=extr.time.dt.dayofyear)
    sd_all_on_extr   = clim_all_daily_std.sel(dayofyear=extr.time.dt.dayofyear)
    base_anom_all    = extr - clim_all_on_extr
    base_stanom_all  = base_anom_all / sd_all_on_extr

    # LOW25 baseline
    clim_low_on_extr = clim_low25_cycle_rm.sel(dayofyear=extr.time.dt.dayofyear)
    sd_low_on_extr   = clim_low25_daily_std.sel(dayofyear=extr.time.dt.dayofyear)
    base_anom_low    = extr - clim_low_on_extr
    base_stanom_low  = base_anom_low / sd_low_on_extr

    # ---- containers ----
    n_ref = len(YEARS_SPECIAL)

    anom_all_ppb     = np.full((n_ref, lev_n, t_len), np.nan)
    anom_low_ppb     = np.full((n_ref, lev_n, t_len), np.nan)
    std_anom_all     = np.full((n_ref, lev_n, t_len), np.nan)
    std_anom_low     = np.full((n_ref, lev_n, t_len), np.nan)

    t_sig_all        = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_all        = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    t_sig_low        = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_low        = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    t_sig_std_all    = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_std_all    = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    t_sig_std_low    = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_std_low    = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    years_hind = np.unique(hind.time.dt.year.values.astype(int))
    print("[DEBUG] hind years:", years_hind)

    for yi, y in enumerate(YEARS_SPECIAL):
        print(f"\n[BlockC_CLOX_v7] -- Special year {y:04d} --")
        if (y not in years_hind) or (y-1 not in years_hind):
            print("  missing y or y-1 in hind, skip")
            continue

        stitched = stitch_prev_oct_to_curr_sep_2d(hind, y)  # (time=365, lev)
        if stitched is None or stitched.sizes["time"] != t_len:
            print("  stitch failed/length mismatch, skip")
            continue

        ref_vals = stitched.transpose("time","lev").values     # (365, lev)
        ref_comp = ref_vals.T                                  # (lev, time)

        # raw anomaly (mol/mol)
        anom_all = ref_comp - clim_all_comp_vals
        anom_low = ref_comp - clim_low_comp_vals

        # standardized anomaly
        stdanom_all = anom_all / std_all_comp_vals
        stdanom_low = anom_low / std_low_comp_vals

        # store ppb
        anom_all_ppb[yi,:,:] = anom_all * 1e9
        anom_low_ppb[yi,:,:] = anom_low * 1e9
        std_anom_all[yi,:,:] = stdanom_all
        std_anom_low[yi,:,:] = stdanom_low

        # significance for RAW anomalies
        print("  sig: RAW vs ALL baseline ...")
        ta, ba = compute_significance_for_baseline(
            base_anom_all, anom_all, doy_extr, doy_comp
        )
        print("  sig: RAW vs LOW25 baseline ...")
        tl, bl = compute_significance_for_baseline(
            base_anom_low, anom_low, doy_extr, doy_comp
        )

        t_sig_all[yi,:,:] = ta
        b_sig_all[yi,:,:] = ba
        t_sig_low[yi,:,:] = tl
        b_sig_low[yi,:,:] = bl

        # significance for STANDARDIZED anomalies
        print("  sig: STD vs ALL baseline ...")
        tsa, bsa = compute_significance_for_baseline(
            base_stanom_all, stdanom_all, doy_extr, doy_comp
        )
        print("  sig: STD vs LOW25 baseline ...")
        tsl, bsl = compute_significance_for_baseline(
            base_stanom_low, stdanom_low, doy_extr, doy_comp
        )

        t_sig_std_all[yi,:,:] = tsa
        b_sig_std_all[yi,:,:] = bsa
        t_sig_std_low[yi,:,:] = tsl
        b_sig_std_low[yi,:,:] = bsl

    # baselines to ppb for output
    clim_all_comp_ppb = clim_all_comp_vals * 1e9
    clim_low_comp_ppb = clim_low_comp_vals * 1e9

    return dict(
        lev=lev, doy_comp=doy_comp, t_len=t_len,
        anom_all_ppb=anom_all_ppb,
        anom_low_ppb=anom_low_ppb,
        std_anom_all=std_anom_all,
        std_anom_low=std_anom_low,
        clim_all_comp_ppb=clim_all_comp_ppb,
        clim_low_comp_ppb=clim_low_comp_ppb,
        t_sig_all=t_sig_all, b_sig_all=b_sig_all,
        t_sig_low=t_sig_low, b_sig_low=b_sig_low,
        t_sig_std_all=t_sig_std_all, b_sig_std_all=b_sig_std_all,
        t_sig_std_low=t_sig_std_low, b_sig_std_low=b_sig_std_low
    )

def main_blockC_CLOX_v7():
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockC_CLOX_v7] Lowest 25% EXTR years:", lowest25_years)

    # load
    extr_pcap = xr.load_dataarray(IN_EXTR_PCAP)
    hind_pcap = xr.load_dataarray(IN_HIND_PCAP)
    extr_82   = xr.load_dataarray(IN_EXTR_82N)
    hind_82   = xr.load_dataarray(IN_HIND_82N)

    out_pcap = process_region(extr_pcap, hind_pcap, lowest25_years, "pcap60_90N")
    out_82   = process_region(extr_82, hind_82, lowest25_years, "82N")

    # coords
    lev = out_pcap["lev"]
    doy_comp = out_pcap["doy_comp"]
    t_len = out_pcap["t_len"]

    ds_out = xr.Dataset(
        coords=dict(
            ref_year=("ref_year", YEARS_SPECIAL),
            lev=("lev", lev),
            time_index=("time_index", np.arange(t_len)),
            dayofyear=("time_index", doy_comp),
        ),
        data_vars=dict(
            # ---- pcap ----
            anom_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["anom_all_ppb"]),
            anom_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["anom_low_ppb"]),
            std_anom_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["std_anom_all"]),
            std_anom_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["std_anom_low"]),
            clim_all_comp_pcap60_90N=(("lev","time_index"), out_pcap["clim_all_comp_ppb"]),
            clim_low25_comp_pcap60_90N=(("lev","time_index"), out_pcap["clim_low_comp_ppb"]),
            t_sig_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["t_sig_all"]),
            b_sig_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["b_sig_all"]),
            t_sig_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["t_sig_low"]),
            b_sig_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["b_sig_low"]),
            t_sig_std_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["t_sig_std_all"]),
            b_sig_std_all_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["b_sig_std_all"]),
            t_sig_std_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["t_sig_std_low"]),
            b_sig_std_low25_pcap60_90N=(("ref_year","lev","time_index"), out_pcap["b_sig_std_low"]),

            # ---- 82N ----
            anom_all_82N=(("ref_year","lev","time_index"), out_82["anom_all_ppb"]),
            anom_low25_82N=(("ref_year","lev","time_index"), out_82["anom_low_ppb"]),
            std_anom_all_82N=(("ref_year","lev","time_index"), out_82["std_anom_all"]),
            std_anom_low25_82N=(("ref_year","lev","time_index"), out_82["std_anom_low"]),
            clim_all_comp_82N=(("lev","time_index"), out_82["clim_all_comp_ppb"]),
            clim_low25_comp_82N=(("lev","time_index"), out_82["clim_low_comp_ppb"]),
            t_sig_all_82N=(("ref_year","lev","time_index"), out_82["t_sig_all"]),
            b_sig_all_82N=(("ref_year","lev","time_index"), out_82["b_sig_all"]),
            t_sig_low25_82N=(("ref_year","lev","time_index"), out_82["t_sig_low"]),
            b_sig_low25_82N=(("ref_year","lev","time_index"), out_82["b_sig_low"]),
            t_sig_std_all_82N=(("ref_year","lev","time_index"), out_82["t_sig_std_all"]),
            b_sig_std_all_82N=(("ref_year","lev","time_index"), out_82["b_sig_std_all"]),
            t_sig_std_low25_82N=(("ref_year","lev","time_index"), out_82["t_sig_std_low"]),
            b_sig_std_low25_82N=(("ref_year","lev","time_index"), out_82["b_sig_std_low"]),
        )
    )

    # attrs
    for v in ds_out.data_vars:
        if v.startswith("std_anom") or v.startswith("t_sig") or v.startswith("b_sig"):
            ds_out[v].attrs["units"] = "1"
        elif v.startswith("anom") or v.startswith("clim"):
            ds_out[v].attrs["units"] = "ppb"

    ds_out.attrs["description"] = (
        "CLOX special-year (8,13,14,19) Oct–Sep stitched time–height anomalies. "
        "Baselines: EXTR all-years and EXTR lowest-25% (prev Oct–Dec + curr Jan–Sep fix). "
        "Seasonal cycle uses 15-day running mean. "
        "Includes raw anomalies (ppb), standardized anomalies, and significance masks "
        "(t-test + bootstrap) for each baseline and anomaly type."
    )

    ds_out.to_netcdf(OUT_NC)
    print("\n[BlockC_CLOX_v7] Saved:", OUT_NC)
    print("[BlockC_CLOX_v7] Done.")

if __name__ == "__main__":
    main_blockC_CLOX_v7()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (CLOX_v7)
#
# Plot time–height cross-sections for BOTH regions:
#   - raw anomalies (ppb)
#   - standardized anomalies
#
# For EACH ref_year:
#   baseline = ALL years  -> t-test & bootstrap masks
#   baseline = LOW25      -> t-test & bootstrap masks
#
# X-axis: day index (0..364), ticks Oct–Sep.
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT_CLOX = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_CLOX"
IN_NC = os.path.join(ROOT_CLOX, "CLOX_vert_anom_special.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "Jun", "Jul", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6.4, 4.2), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)

def plot_time_height(
    x_idx,
    lev_hpa,
    field,          # (lev, time)
    clim_field,     # (lev, time) baseline mean (ppb) for contour overlay; can be None
    nonsig_mask,    # (lev, time) True = NOT significant
    ref_year,
    region_title,
    baseline_label,
    field_label,    # "anom" or "std_anom"
    test_label,
    outfile,
):
    

    fig, ax = plt.subplots()

    x = np.asarray(x_idx)
    p = np.asarray(lev_hpa)

    # levels
    if field_label == "std_anom":
        vmax = 3.0
        levels = np.linspace(-vmax, vmax, 13)
    else:
        # ---------- 🔧 强制固定色标范围 ----------
        vmin, vmax = -1.3, 1.3
        levels = np.linspace(vmin, vmax, 31)

    cf = ax.contourf(
        x, p, field,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.9
    )

    # optional baseline contours (only for raw anom; std no need)
    if (clim_field is not None) and (field_label != "std_anom"):
        clim_mean = np.nanmean(clim_field)
        clim_std  = np.nanstd(clim_field)
        if np.isfinite(clim_mean) and np.isfinite(clim_std) and clim_std > 0:
            clim_levels = np.linspace(clim_mean-1.5*clim_std, clim_mean+1.5*clim_std, 7)
        else:
            clim_levels = 7

        CS = ax.contour(
            x, p, clim_field,
            levels=clim_levels,
            colors="k",
            linewidths=0.6,
        )
        ax.clabel(CS, inline=True, fontsize=6, fmt="%.1f")

    # hatch nonsig
    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
        )
        patch = Patch(facecolor="white", hatch="///", edgecolor="black",
                      label="Not significant (p ≥ 0.05)")
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel("Pressure (hPa)")
    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    title_main = "CLOX anomaly (ppb)" if field_label=="anom" else "CLOX standardized anomaly"
    ax.set_title(
        f"{title_main}\n"
        f"{region_title}, Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("Std. anom" if field_label=="std_anom" else "Anomaly (ppb)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)

def main_blockD_CLOX_v7():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    regions = [
        dict(
            tag="82N",
            title="82°N zonal mean",
            anom_all=ds["anom_all_82N"].values,
            anom_low=ds["anom_low25_82N"].values,
            std_all=ds["std_anom_all_82N"].values,
            std_low=ds["std_anom_low25_82N"].values,
            clim_all=ds["clim_all_comp_82N"].values,
            clim_low=ds["clim_low25_comp_82N"].values,
            t_all=ds["t_sig_all_82N"].values,
            b_all=ds["b_sig_all_82N"].values,
            t_low=ds["t_sig_low25_82N"].values,
            b_low=ds["b_sig_low25_82N"].values,
            t_std_all=ds["t_sig_std_all_82N"].values,
            b_std_all=ds["b_sig_std_all_82N"].values,
            t_std_low=ds["t_sig_std_low25_82N"].values,
            b_std_low=ds["b_sig_std_low25_82N"].values,
        ),
        dict(
            tag="pcap60_90N",
            title="polar cap 60–90°N weighted mean",
            anom_all=ds["anom_all_pcap60_90N"].values,
            anom_low=ds["anom_low25_pcap60_90N"].values,
            std_all=ds["std_anom_all_pcap60_90N"].values,
            std_low=ds["std_anom_low25_pcap60_90N"].values,
            clim_all=ds["clim_all_comp_pcap60_90N"].values,
            clim_low=ds["clim_low25_comp_pcap60_90N"].values,
            t_all=ds["t_sig_all_pcap60_90N"].values,
            b_all=ds["b_sig_all_pcap60_90N"].values,
            t_low=ds["t_sig_low25_pcap60_90N"].values,
            b_low=ds["b_sig_low25_pcap60_90N"].values,
            t_std_all=ds["t_sig_std_all_pcap60_90N"].values,
            b_std_all=ds["b_sig_std_all_pcap60_90N"].values,
            t_std_low=ds["t_sig_std_low25_pcap60_90N"].values,
            b_std_low=ds["b_sig_std_low25_pcap60_90N"].values,
        ),
    ]

    ds.close()

    for reg in regions:
        print(f"\n[BlockD_CLOX_v7] ===== Region {reg['tag']} =====")

        for yi, y in enumerate(ref_years):

            # ---------- RAW anomalies ----------
            # vs ALL baseline
            for test, sig in [("t-test", reg["t_all"]), ("bootstrap", reg["b_all"])]:
                nonsig = ~sig[yi,:,:]
                outfile = os.path.join(
                    ROOT_CLOX,
                    f"CLOX_anom_{reg['tag']}_year{int(y):04d}_vsALL_{test.replace('-','')}.png"
                )
                plot_time_height(
                    time_index, lev,
                    reg["anom_all"][yi,:,:],
                    reg["clim_all"],
                    nonsig,
                    y, reg["title"],
                    "EXTR all-years climatology",
                    "anom",
                    test,
                    outfile
                )

            # vs LOW25 baseline
            for test, sig in [("t-test", reg["t_low"]), ("bootstrap", reg["b_low"])]:
                nonsig = ~sig[yi,:,:]
                outfile = os.path.join(
                    ROOT_CLOX,
                    f"CLOX_anom_{reg['tag']}_year{int(y):04d}_vsLOW25_{test.replace('-','')}.png"
                )
                plot_time_height(
                    time_index, lev,
                    reg["anom_low"][yi,:,:],
                    reg["clim_low"],
                    nonsig,
                    y, reg["title"],
                    "EXTR low-25% climatology",
                    "anom",
                    test,
                    outfile
                )

            # ---------- STANDARDIZED anomalies ----------
            # vs ALL baseline
            for test, sig in [("t-test", reg["t_std_all"]), ("bootstrap", reg["b_std_all"])]:
                nonsig = ~sig[yi,:,:]
                outfile = os.path.join(
                    ROOT_CLOX,
                    f"CLOX_stdanom_{reg['tag']}_year{int(y):04d}_vsALL_{test.replace('-','')}.png"
                )
                plot_time_height(
                    time_index, lev,
                    reg["std_all"][yi,:,:],
                    None,  # no baseline contour for std_anom
                    nonsig,
                    y, reg["title"],
                    "EXTR all-years climatology",
                    "std_anom",
                    test,
                    outfile
                )

            # vs LOW25 baseline
            for test, sig in [("t-test", reg["t_std_low"]), ("bootstrap", reg["b_std_low"])]:
                nonsig = ~sig[yi,:,:]
                outfile = os.path.join(
                    ROOT_CLOX,
                    f"CLOX_stdanom_{reg['tag']}_year{int(y):04d}_vsLOW25_{test.replace('-','')}.png"
                )
                plot_time_height(
                    time_index, lev,
                    reg["std_low"][yi,:,:],
                    None,
                    nonsig,
                    y, reg["title"],
                    "EXTR low-25% climatology",
                    "std_anom",
                    test,
                    outfile
                )

    print("\n[BlockD_CLOX_v7] All figures generated.")

if __name__ == "__main__":
    main_blockD_CLOX_v7()
